# 05c · Yeni ana model C (IR-telafili hibrit) — kayıt + sağlamlık + verimlilik (EĞİTİM YOK, ~1 saat)
Karar (03.11.2026): ana model = **yalnız ohmik kol + akım omurgaya da girer** (03b'deki "RC: yalnız ohmik + akım girişi").
1. 03b'de eğitilen 5 seed **kopyalanır** (`final_c` aşaması, ad: *Önerilen (IR-telafili hibrit)*) ve zaman serileriyle yeniden değerlendirilir (Şekil 3–5 için).
2. Sağlamlık: 05 ile **aynı bozulmalar, aynı tohumlar, aynı 3 seed** → mevcut 05 satırlarıyla eşleştirilmiş karşılaştırma. Çok ölçekli uzantı (RC + akım girişi) da eklenir.
3. Verimlilik (parametre, MAC, GPU/CPU gecikme) ve sızıntı–hata ilişkisi.
Önce 03b tamamlanmış olmalı. Sonra **03c** (eğitim) çalıştırılır.

In [ ]:
import sys, os, json, time, warnings
import os; ROOT = os.path.abspath('..')  # repository root (run notebooks from the notebooks/ folder)
SMOKE = False                 # True: çok küçük hızlı deneme (kurulum testi)
sys.path.insert(0, ROOT)
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch
from data_utils import *
from models.experiment import *
from models import engine
exp = Experiment(ROOT, smoke=SMOKE)
exp.load_data()
print('Cihaz:', exp.dev, '| W =', exp.cfg['W'])

In [ ]:
t0 = time.time()
register_c(exp, exp.cfg['FINAL_SEEDS'])
fc = exp.collect('final_c')
t = (fc.assign(k=np.where(fc.temp == '-20degC', '−20°C', 'eğitim sıc.')).groupby(['k', 'seed']).rmse.mean() * 100).groupby('k').agg(['mean', 'std'])
print(t.round(3)); print(f'kayıt + seriler {(time.time() - t0) / 60:.1f} dk')

In [ ]:
# Sağlamlık (05 ile eşleştirilmiş: aynı senaryo tohumları 1000+si, aynı seed'ler)
ROB_SEEDS = exp.cfg['FINAL_SEEDS'][:3]
WHO = [(C_NAME, 'final_c'), ('RC + akım girişi', 'ablation_b')]
out_csv = exp.p['excel'] / '05c_saglamlik_satirlar.csv'
done = pd.read_csv(out_csv) if out_csv.exists() else pd.DataFrame(columns=['senaryo', 'model', 'seed'])
rows = [] if not len(done) else done.to_dict('records')
for si, (scen, sc) in enumerate(ROBUST_SCENARIOS.items()):
    dfs = {k: perturb(d, sc, seed=1000 + si) for k, d in exp.te_dfs.items()}
    pad = 'start_soc' in sc
    for name, stage in WHO:
        for seed in ROB_SEEDS:
            if len(done) and ((done.senaryo == scen) & (done.model == name) & (done.seed == seed)).any(): continue
            m, rec = exp.load_model(stage, name, seed)
            r, _ = engine.evaluate(m, exp.test_sets_from(dfs, rec['spec']['window'], pad), dfs, exp.cfg)
            rows += [{'senaryo': scen, 'model': name, 'seed': seed, **x} for x in r]
            del m; torch.cuda.empty_cache()
    pd.DataFrame(rows).to_csv(out_csv, index=False); print(f'✅ {scen}')
rob = pd.DataFrame(rows)
old = exp.p['excel'] / '05_saglamlik_satirlar.csv'
allr = pd.concat([pd.read_csv(old), rob]) if old.exists() else rob
allr['k'] = np.where(allr.temp == '-20degC', '−20', 'görülen')
tab = (allr.groupby(['senaryo', 'model', 'k', 'seed']).rmse.mean() * 100).groupby(['senaryo', 'model', 'k']).mean().unstack(['model', 'k']).reindex(list(ROBUST_SCENARIOS))
print(tab.round(2).to_string())

In [ ]:
# Eşleştirilmiş karşılaştırma: C vs her model, senaryo başına −20 °C ve görülen (Wilcoxon, seed×dosya)
from scipy import stats
cell = allr.groupby(['senaryo', 'model', 'seed', 'temp', 'cycle']).rmse.mean() * 100
sig = []
for scen in ROBUST_SCENARIOS:
    for other in [m for m in allr.model.unique() if m != C_NAME]:
        for k, f in [('−20', lambda i: i.get_level_values('temp') == '-20degC'), ('görülen', lambda i: i.get_level_values('temp') != '-20degC')]:
            a = cell.loc[(scen, C_NAME)]; a = a[f(a.index)]
            b = cell.loc[(scen, other)]
            if other == '2RC-ECM + EKF':
                a = a.groupby(['temp', 'cycle']).mean(); b = b.droplevel('seed')
            b = b.reindex(a.index); d = (a - b).dropna()
            if len(d): sig.append({'senaryo': scen, 'karşı': other, 'kapsam': k, 'Δ ort (C − model)': d.mean(), 'C daha iyi': f'{(d < 0).sum()}/{len(d)}',
                                   'Wilcoxon p': stats.wilcoxon(d).pvalue if (d != 0).any() and len(d) > 1 else np.nan})
sig = pd.DataFrame(sig); print(sig[sig.kapsam == '−20'].round(4).to_string(index=False))

In [ ]:
# Verimlilik (02 ile aynı ölçüm) + sızıntı–hata ilişkisi (C'nin en iyi val seed serisi)
m, rec = exp.load_model('final_c', C_NAME, exp.cfg['FINAL_SEEDS'][0])
X = next(iter(exp.datasets(rec['spec']['window'])['test'].values())).X[:rec['spec']['window']].unsqueeze(0)
e = engine.efficiency(m, X, n_rep=50 if exp.smoke else 200)
rr = exp.records('final_c')
e.update({'model': C_NAME, 'W (s)': rec['spec']['window'], 'eğitim süresi ort (dk)': np.mean([r['train']['train_time_s'] for r in rr]) / 60,
          'epoch süresi ort (s)': np.mean([r['train']['epoch_time_mean_s'] for r in rr]),
          'eğitim GPU bellek tepe (GB)': np.mean([r['train'].get('train_peak_mem_gb', np.nan) for r in rr]), 'ön işleme (ms / örnek)': exp.pre_ms})
pd.DataFrame([e]).to_json(exp.p['configs'] / 'verimlilik_c.json', orient='records', force_ascii=False, indent=1)
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in e.items()})
rho = None
ap = exp.p['excel'] / '05_sizinti_pencereler.csv'
if ap.exists():
    aud = pd.read_csv(ap); best = min(rr, key=lambda r: r['val_rmse_final'])
    ser = load_table(exp.paths('final_c', C_NAME, best['seed'])[2]); errs = []
    for (t, c), g in aud.groupby(['temp', 'cycle'], sort=False):
        ee = ser[(ser.temp == t) & (ser.cycle == c)].err.values
        errs += [float(np.sqrt(np.mean(ee[a:a + 60] ** 2))) if len(ee[a:a + 60]) else np.nan for a in g.start.values]
    aud = pd.concat([g for _, g in aud.groupby(['temp', 'cycle'], sort=False)]); aud['C_rmse_60s'] = errs
    rho = aud.groupby('temp').apply(lambda g: stats.spearmanr(g.max_corr, g.C_rmse_60s, nan_policy='omit').correlation)
    print('Spearman (benzerlik, C hatası):'); print(rho.round(3).to_string())
with pd.ExcelWriter(exp.p['excel'] / '05c_anamodel_C.xlsx') as xw:
    fc.to_excel(xw, sheet_name='C final satırlar', index=False)
    tab.to_excel(xw, sheet_name='Sağlamlık RMSE'); sig.to_excel(xw, sheet_name='Sağlamlık anlamlılık', index=False)
    pd.DataFrame([e]).to_excel(xw, sheet_name='Verimlilik', index=False)
    if rho is not None: rho.rename('spearman').to_frame().to_excel(xw, sheet_name='Sızıntı-hata')
print('✅', exp.p['excel'] / '05c_anamodel_C.xlsx')